# BMP — chạy thực nghiệm trên Google Colab, 100 giây/cấu hình

Mở notebook này bằng **Upload notebook** trên https://colab.research.google.com/ và chọn runtime CPU.
Chạy lần lượt các ô. Mã hiện tại dùng CPU, không gọi GPU.

Chuẩn bị file **colab_source.zip** cùng notebook từ máy của bạn. ZIP chứa `src/`, `data/`, `scripts/run_solver.py` và `requirements.txt` ở gốc; không cần môi trường Windows `.venv`. Bản ZIP được tạo từ mã nguồn hiện tại, gồm các file chưa commit. Nếu sửa thuật toán, cần tạo ZIP mới.

Mặc định: 100 giây tổng thời gian worker cho mỗi cấu hình/instance, một luồng; 4 cấu hình SAT + CP-SAT + 8 cấu hình AMO mở rộng = **13 cấu hình**, tái dùng hai cấu hình pairwise-binomial cho cả so sánh SAT và AMO. Chạy từng đợt 5 instance và lưu CSV sau mỗi cấu hình lên Google Drive.

Giới hạn 100 giây không giới hạn RAM. Runtime Colab có thể dừng trước khi hoàn thành toàn bộ 113 instance; chạy lại các ô thiết lập sau khi nối lại runtime và chạy đợt tiếp theo. Không cần để laptop tự giải bài toán.

**1. Cài các thư viện cho SAT và CP-SAT.** Solver thương mại được cấu hình riêng ở cuối notebook.

In [ ]:
%pip install -q "python-sat>=1.9.dev15" "ortools>=9.15"

**2. Kết nối Drive và upload colab_source.zip.** Cấp quyền Drive trong hộp thoại Colab. Source được giải nén trên ổ tạm Colab; CSV được ghi lên Drive.

In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile

drive.mount("/content/drive")

# Upload colab_source.zip prepared from the CURRENT local source.
# This avoids cloning an older GitHub commit that lacks uncommitted files.
uploaded = files.upload()
archive_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(archive_names) != 1:
    raise ValueError("Upload exactly one colab_source.zip file.")
ROOT = Path("/content/bmp_project")
ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive_names[0]) as archive:
    for item in archive.infolist():
        target = (ROOT / item.filename).resolve()
        if not target.is_relative_to(ROOT.resolve()):
            raise ValueError("Unsafe archive path: " + item.filename)
    archive.extractall(ROOT)
del uploaded
assert (ROOT / "src/search/repeated.py").is_file(), "Archive must contain src/ and data/ at its root."
assert len(list((ROOT / "data").rglob("*.mtx.rnd"))) == 113
print("Project:", ROOT)


**3. Chọn giới hạn và số instance mỗi đợt.** `BATCH_SIZE=5` là số instance chưa hoàn thành, không phải chỉ chạy 5 cấu hình. Đặt `None` nếu muốn chạy mọi instance còn lại trong phiên hiện tại. `INCLUDE_AMO=False` chạy riêng bốn cấu hình SAT và CP-SAT.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(ROOT))
TIME_LIMIT = 100.0          # wall-clock seconds per configuration per instance
BATCH_SIZE = 5              # instances per notebook run; None selects every remaining instance
INCLUDE_AMO = True          # pairwise + binary/commander/product/sequential
SOLVER_METHODS = ["cpsat"]  # add cplex_cp/cplex_mip/gurobi only after engine/license setup

OUTPUT = Path("/content/drive/MyDrive/BMP_results/benchmark_100s.csv")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
print("Results:", OUTPUT)


**4. Khởi tạo worker và bộ chạy.** Worker nằm trong thư mục tạm, không cần khôi phục các script benchmark cũ. Nó bị dừng khi hết 100 giây; các nghiệm đã gửi được kiểm tra độc lập bằng verify.py.

In [ ]:
WORKER_SOURCE = "\"\"\"One isolated Colab benchmark configuration, with progress saved as JSONL.\"\"\"\nimport argparse\nimport json\nfrom pathlib import Path\nimport sys\nfrom time import perf_counter\n\nparser = argparse.ArgumentParser()\nparser.add_argument(\"--root\", required=True)\nparser.add_argument(\"--instance\", required=True)\nparser.add_argument(\"--config\", required=True)\nparser.add_argument(\"--events\", required=True)\nparser.add_argument(\"--deadline\", type=float, required=True)\nargs = parser.parse_args()\nsys.path.insert(0, args.root)\n\nwith open(args.events, \"a\", encoding=\"utf-8\", buffering=1) as stream:\n    def emit(event):\n        event[\"at\"] = perf_counter()\n        stream.write(json.dumps(event, allow_nan=False) + \"\\n\")\n        stream.flush()\n\n    try:\n        from src.graph import load_graph\n        graph = load_graph(args.instance)\n        config = json.loads(args.config)\n        if config[\"kind\"] == \"sat\":\n            from src.search.repeated import optimize_repeated\n            from src.search.incremental import optimize_incremental\n            best = graph.n\n\n            def on_trial(trial, positions):\n                global best\n                improved = positions is not None and trial.bandwidth < best\n                if improved:\n                    best = trial.bandwidth\n                emit({\n                    \"type\": \"trial\", \"k\": trial.k, \"sat\": trial.satisfiable,\n                    \"positions\": positions if improved else None,\n                    \"variables\": trial.num_variables, \"clauses\": trial.num_clauses,\n                    \"build\": trial.build_seconds, \"solve\": trial.solve_seconds,\n                })\n\n            method = {\"repeated\": optimize_repeated, \"incremental\": optimize_incremental}[config[\"method\"]]\n            result = method(graph, config[\"encoding\"], on_trial=on_trial, cardinality=config[\"cardinality\"])\n            emit({\n                \"type\": \"result\", \"status\": \"OPTIMAL\",\n                \"positions\": result.positions, \"bandwidth\": result.bandwidth,\n                \"lower_bound\": result.bandwidth, \"build\": result.build_seconds,\n                \"solve\": result.solve_seconds,\n            })\n        else:\n            from src.solvers.cpsat import solve_cpsat\n            from src.solvers.cplex_cp import solve_cplex_cp\n            from src.solvers.cplex_mip import solve_cplex_mip\n            from src.solvers.gurobi import solve_gurobi\n            methods = {\"cpsat\": solve_cpsat, \"cplex_cp\": solve_cplex_cp,\n                       \"cplex_mip\": solve_cplex_mip, \"gurobi\": solve_gurobi}\n            remaining = args.deadline - perf_counter() - 0.5\n            if remaining <= 0:\n                emit({\"type\": \"result\", \"status\": \"TIMEOUT\", \"positions\": None,\n                      \"bandwidth\": None, \"lower_bound\": None, \"build\": 0.0, \"solve\": 0.0})\n            else:\n                result = methods[config[\"method\"]](graph, time_limit=remaining, workers=1)\n                emit({\n                    \"type\": \"result\", \"status\": result.status,\n                    \"positions\": result.positions, \"bandwidth\": result.bandwidth,\n                    \"lower_bound\": result.lower_bound, \"build\": result.build_seconds,\n                    \"solve\": result.solve_seconds,\n                })\n    except Exception as exc:\n        emit({\"type\": \"error\", \"message\": f\"{type(exc).__name__}: {exc}\"})\n        sys.exit(1)\n"

from pathlib import Path
import sys
import csv
import hashlib
import importlib.metadata
import json
import math
import os
import platform
import signal
import socket
import subprocess
import tempfile
from datetime import datetime, timezone
from time import perf_counter

from src.graph import list_instance_paths, load_graph
from src.verify import verify_solution
from src.encodings.cardinality import encode_exactly_one

WORKER = Path(tempfile.gettempdir()) / "bmp_colab_worker.py"
WORKER.write_text(WORKER_SOURCE, encoding="utf-8")

FIELDS = (
    "case_id", "session_id", "started_utc", "source", "instance", "n", "m",
    "kind", "method", "encoding", "cardinality", "status", "bandwidth",
    "lower_bound", "initial_upper_bound", "positions", "last_k", "sat_calls",
    "num_variables", "num_clauses", "primary_variables", "aux_variables",
    "assignment_clauses", "build_seconds", "solve_seconds", "wall_seconds",
    "time_limit_seconds", "workers", "code_hash", "python_version", "packages",
    "cpu", "logical_cpus", "platform", "host", "error",
)

def cpu_name():
    info = Path("/proc/cpuinfo")
    if info.is_file():
        for line in info.read_text().splitlines():
            if line.startswith("model name"):
                return line.split(":", 1)[1].strip()
    return platform.processor()

def environment():
    packages = {}
    for name in ("python-sat", "ortools", "docplex", "cplex", "gurobipy"):
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    digest = hashlib.sha256(WORKER_SOURCE.encode())
    for path in sorted((ROOT / "src").rglob("*.py")):
        digest.update(path.relative_to(ROOT).as_posix().encode())
        digest.update(path.read_bytes())
    return {
        "code_hash": digest.hexdigest(), "python_version": platform.python_version(),
        "packages": json.dumps(packages, sort_keys=True), "cpu": cpu_name(),
        "logical_cpus": os.cpu_count(), "platform": platform.platform(),
        "host": socket.gethostname(),
    }

def configurations():
    choices = [
        {"kind": "sat", "method": method, "encoding": encoding, "cardinality": "binomial"}
        for encoding in ("pairwise", "order")
        for method in ("repeated", "incremental")
    ]
    if INCLUDE_AMO:
        choices += [
            {"kind": "sat", "method": method, "encoding": "pairwise", "cardinality": amo}
            for amo in ("binary", "commander", "product", "sequential")
            for method in ("repeated", "incremental")
        ]
    choices += [
        {"kind": "solver", "method": method, "encoding": "", "cardinality": ""}
        for method in SOLVER_METHODS
    ]
    valid = {"cpsat", "cplex_cp", "cplex_mip", "gurobi"}
    if len(set(SOLVER_METHODS)) != len(SOLVER_METHODS) or set(SOLVER_METHODS) - valid:
        raise ValueError("SOLVER_METHODS contains duplicate or unsupported methods.")
    return choices

def case_id(path, config, env, time_limit):
    # Changing data, source, dependencies, Python, or time budget creates a new case.
    payload = {
        "source": path.relative_to(ROOT).as_posix(),
        "data_hash": hashlib.sha256(path.read_bytes()).hexdigest(),
        "config": config, "time_limit": time_limit, "workers": 1,
        "code_hash": env["code_hash"], "python_version": env["python_version"],
        "packages": env["packages"],
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()

def completed_cases(output):
    if not output.is_file() or not output.stat().st_size:
        return set()
    with output.open(encoding="utf-8", newline="") as stream:
        reader = csv.DictReader(stream)
        if tuple(reader.fieldnames or ()) != FIELDS:
            raise ValueError("Existing CSV has a different schema. Choose a new OUTPUT path.")
        return {
            row["case_id"] for row in reader
            if row.get("status") in ("OPTIMAL", "FEASIBLE", "TIMEOUT")
            and row.get("error") == ""
            and row.get("wall_seconds") not in (None, "")
        }

def _append_csv(output, row):
    output.parent.mkdir(parents=True, exist_ok=True)
    fresh = not output.is_file() or not output.stat().st_size
    with output.open("a", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=FIELDS)
        if fresh:
            writer.writeheader()
        writer.writerow(row)
        stream.flush()
        os.fsync(stream.fileno())

def comparison_groups(row):
    groups = []
    baseline_sat = row["kind"] == "sat" and row["cardinality"] == "binomial"
    if baseline_sat:
        groups.append("encoding")
    if baseline_sat or row["kind"] == "solver":
        groups.append("solver")
    if row["kind"] == "sat" and row["encoding"] == "pairwise":
        groups.append("cardinality")
    return groups

def comparison_paths(output):
    paths = {name: output.parent / (name + ".csv")
             for name in ("encoding", "solver", "cardinality")}
    if output.resolve() in {path.resolve() for path in paths.values()}:
        raise ValueError("OUTPUT must be the checkpoint CSV, not a comparison CSV.")
    return paths

def export_comparison_csvs(output):
    """Rebuild all three comparison files from saved results, without solving again."""
    paths = comparison_paths(output)
    rows = []
    if output.is_file() and output.stat().st_size:
        with output.open(encoding="utf-8", newline="") as stream:
            reader = csv.DictReader(stream)
            if tuple(reader.fieldnames or ()) != FIELDS:
                raise ValueError("Checkpoint CSV has a different schema.")
            rows = list(reader)
    output.parent.mkdir(parents=True, exist_ok=True)
    for name, path in paths.items():
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(
                mode="w", encoding="utf-8", newline="", dir=path.parent,
                prefix=path.name + ".", suffix=".tmp", delete=False,
            ) as stream:
                temporary = Path(stream.name)
                writer = csv.DictWriter(stream, fieldnames=FIELDS)
                writer.writeheader()
                writer.writerows(row for row in rows if name in comparison_groups(row))
                stream.flush()
                os.fsync(stream.fileno())
            os.replace(temporary, path)
        finally:
            if temporary is not None:
                temporary.unlink(missing_ok=True)
    return paths

def append_result(output, row):
    paths = comparison_paths(output)
    _append_csv(output, row)
    for name in comparison_groups(row):
        _append_csv(paths[name], row)

def run_case(path, config, env, session_id, time_limit):
    graph = load_graph(path)
    positions = tuple(range(1, graph.n + 1))
    baseline = verify_solution(graph, positions)
    row = dict.fromkeys(FIELDS, "")
    row.update(env)
    row.update(config)
    row.update(
        case_id=case_id(path, config, env, time_limit), session_id=session_id,
        started_utc=datetime.now(timezone.utc).isoformat(),
        source=path.relative_to(ROOT).as_posix(), instance=graph.name, n=graph.n, m=graph.m,
        status="TIMEOUT", bandwidth=baseline, lower_bound=1 if graph.m else 0,
        initial_upper_bound=baseline, positions=json.dumps(positions),
        sat_calls=0, build_seconds=0.0, solve_seconds=0.0,
        time_limit_seconds=time_limit, workers=1,
    )
    if config["encoding"] == "pairwise":
        sample = encode_exactly_one(tuple(range(1, graph.n + 1)), config["cardinality"])
        row.update(
            primary_variables=graph.n ** 2,
            aux_variables=2 * graph.n * (sample.num_vars - graph.n),
            assignment_clauses=2 * graph.n * len(sample.clauses),
        )

    with tempfile.TemporaryDirectory(prefix="bmp-colab-") as folder:
        events_path = Path(folder) / "events.jsonl"
        start = perf_counter()
        deadline = start + time_limit
        command = [
            sys.executable, str(WORKER), "--root", str(ROOT), "--instance", str(path),
            "--config", json.dumps(config), "--events", str(events_path),
            "--deadline", str(deadline),
        ]
        process = subprocess.Popen(
            command, cwd=ROOT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
            start_new_session=(os.name != "nt"),
        )
        timed_out = False
        try:
            process.wait(timeout=max(0.001, deadline - perf_counter()))
        except subprocess.TimeoutExpired:
            timed_out = True
            if os.name == "nt":
                process.kill()
            else:
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
            process.wait()
        except BaseException:
            # Stop the current worker if the notebook cell is interrupted.
            if process.poll() is None:
                if os.name == "nt":
                    process.kill()
                else:
                    try:
                        os.killpg(process.pid, signal.SIGKILL)
                    except ProcessLookupError:
                        pass
                process.wait()
            raise
        row["wall_seconds"] = perf_counter() - start
        events = []
        if events_path.is_file():
            for line in events_path.read_text(encoding="utf-8").splitlines():
                try:
                    event = json.loads(line)
                except json.JSONDecodeError:
                    continue
                if event.get("at", float("inf")) <= deadline:
                    events.append(event)

    try:
        for event in events:
            if event["type"] == "error":
                raise ValueError(event["message"])
            if event.get("positions") is not None:
                candidate = tuple(event["positions"])
                actual = verify_solution(graph, candidate)
                if event["type"] == "result" and event["status"] == "OPTIMAL":
                    verify_solution(graph, candidate, event["bandwidth"])
                if actual < row["bandwidth"]:
                    row["positions"], row["bandwidth"] = json.dumps(candidate), actual
            if event["type"] == "trial":
                row["sat_calls"] += 1
                row.update(last_k=event["k"], num_variables=event["variables"], num_clauses=event["clauses"])
                row["build_seconds"] += event["build"]
                row["solve_seconds"] += event["solve"]
                if not event["sat"]:
                    row["lower_bound"] = max(row["lower_bound"], event["k"] + 1)
            elif event["type"] == "result":
                status = event["status"]
                if status not in ("OPTIMAL", "FEASIBLE", "TIMEOUT"):
                    raise ValueError("Unexpected result status")
                if status in ("OPTIMAL", "FEASIBLE") and event.get("positions") is None:
                    raise ValueError("Result has no witness")
                if event.get("lower_bound") is not None:
                    bound = event["lower_bound"]
                    if not math.isfinite(bound):
                        raise ValueError("Non-finite lower bound")
                    row["lower_bound"] = max(row["lower_bound"], bound)
                row.update(status=status, build_seconds=event["build"], solve_seconds=event["solve"])
                if status == "OPTIMAL":
                    if row["lower_bound"] > row["bandwidth"] or actual != row["bandwidth"]:
                        raise ValueError("Optimal result conflicts with a verified bound")
                    row["lower_bound"] = row["bandwidth"]
        if row["lower_bound"] > row["bandwidth"]:
            raise ValueError("Lower bound exceeds verified bandwidth")
        if process.returncode != 0 and not timed_out:
            raise ValueError("Worker exited with code " + str(process.returncode))
        if not timed_out and not any(event["type"] == "result" for event in events):
            raise ValueError("Worker stopped without a final result")
        verify_solution(graph, json.loads(row["positions"]), row["bandwidth"])
    except (KeyError, TypeError, ValueError) as exc:
        row.update(status="ERROR", error=str(exc))
    return row

def run_batch(paths=None):
    if not math.isfinite(TIME_LIMIT) or TIME_LIMIT <= 0:
        raise ValueError("TIME_LIMIT must be finite and positive.")
    if BATCH_SIZE is not None and (type(BATCH_SIZE) is not int or BATCH_SIZE < 1):
        raise ValueError("BATCH_SIZE must be a positive integer or None.")
    env = environment()
    configs = configurations()
    export_comparison_csvs(OUTPUT)
    done = completed_cases(OUTPUT)
    paths = list(paths if paths is not None else list_instance_paths(ROOT / "data"))
    pending = [
        path for path in paths
        if any(case_id(path, config, env, TIME_LIMIT) not in done for config in configs)
    ]
    selected = pending if BATCH_SIZE is None else pending[:BATCH_SIZE]
    session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    print(f"Pending instances: {len(pending)}; this batch: {len(selected)}; configurations: {len(configs)}", flush=True)
    errors = 0
    for path in selected:
        for config in configs:
            key = case_id(path, config, env, TIME_LIMIT)
            if key in done:
                continue
            row = run_case(path, config, env, session_id, TIME_LIMIT)
            append_result(OUTPUT, row)
            if row["status"] != "ERROR":
                done.add(key)
            else:
                errors += 1
            print(
                f"{row['instance']} | {row['method']} | {row['encoding']} | {row['cardinality']} | "
                f"{row['status']} | B={row['bandwidth']} | LB={row['lower_bound']} | {row['wall_seconds']:.2f}s",
                flush=True,
            )
            if row["error"]:
                print("  " + row["error"], flush=True)
    print(f"Batch complete. Errors: {errors}. Results: {OUTPUT}", flush=True)


**5. Chạy một đợt.** Bấm chạy lại ô này để chuyển sang đợt tiếp theo. Sau khi Colab cấp runtime mới, chạy lại các ô 1–4 rồi ô này.

CSV bỏ qua các cấu hình OPTIMAL/FEASIBLE/TIMEOUT đã lưu với cùng mã nguồn, dữ liệu, Python, thư viện và thời hạn. ERROR được thử lại. Một cấu hình đang chạy khi runtime bị ngắt có thể cần chạy lại. Giữ cùng OUTPUT để tiếp tục; đổi OUTPUT khi muốn thực hiện một lượt đo lặp mới.

Metadata ghi CPU, máy và phiên chạy; phần cứng Colab có thể đổi giữa các phiên, nên phân nhóm phù hợp khi phân tích thời gian. Nếu package/Python thay đổi, các case được coi là lượt đo mới.

Đây là bước tạo CSV, chưa vẽ hình. CSV gồm thời gian, trạng thái, B/LB, hoán vị, số biến/clause và cấu hình môi trường. TIMEOUT giữ nghiệm tốt nhất đã nhận, hoặc nghiệm ban đầu; số dựng/giải có thể thiếu công việc đang chạy khi worker bị ngắt.

In [ ]:
run_batch()

**6. Đọc ba CSV theo nhóm so sánh.** `encoding.csv` chứa bốn cấu hình SAT; `solver.csv` chứa bốn cấu hình SAT cùng các solver CP/MIP đã bật; `cardinality.csv` chứa năm cách AMO × repeated/incremental trên pairwise. Hai cấu hình pairwise-binomial xuất hiện trong cả nhóm encoding và AMO. File `benchmark_100s.csv` giữ lịch sử đầy đủ để tiếp tục; ba file so sánh được cập nhật sau mỗi cấu hình. Ô dưới cũng tách được lịch sử CSV cũ mà không giải lại.

In [ ]:
import pandas as pd

paths = export_comparison_csvs(OUTPUT)
for name, path in paths.items():
    print(name, path)
    results = pd.read_csv(path)
    display(results.groupby(["kind", "method", "encoding", "cardinality", "status"], dropna=False).size().rename("runs").reset_index())
    display(results.tail(5))

**Solver thương mại (tùy chọn).**

- `gurobipy` cài bằng pip có license giới hạn kích thước. Muốn chạy toàn bộ dataset cần license phù hợp; Colab có thể dùng Gurobi WLS. Thiết lập license cho tiến trình con, ví dụ qua file `gurobi.lic` và biến môi trường `GRB_LICENSE_FILE`; không ghi khóa bí mật vào notebook được chia sẻ.
- CPLEX MIP cần `docplex`, `cplex` và license đủ kích thước. Bản Community giới hạn 1000 biến/1000 ràng buộc.
- CPLEX CP cần executable **CP Optimizer dành cho Linux** ngoài `docplex`; đặt `os.environ["CP_OPTIMIZER_EXECUTABLE"]` thành đường dẫn engine đã cài. Không dùng `cpoptimizer.exe` của Windows trên runtime Linux.
- Khi cài đặt xong, sửa ô cấu hình thành `SOLVER_METHODS=["cpsat","cplex_cp","cplex_mip","gurobi"]` và chạy lại ô 4–5. Thay đổi thư viện làm thay đổi case ID để giữ các lượt đo có môi trường nhất quán.
- Để so sánh thời gian công bằng, chạy các phương pháp trên cùng loại CPU, cùng số luồng và thời hạn; metadata giúp nhận biết các phiên Colab khác phần cứng.

Nguồn: [Colab FAQ](https://research.google.com/colaboratory/faq.html), [Gurobi trên Colab](https://support.gurobi.com/hc/en-us/articles/4409582394769-Google-Colab-Installation-and-Licensing), [IBM DOcplex engine setup](https://ibmdecisionoptimization.github.io/docplex-doc/getting_started.html).

`run_solver.py` cũng mặc định 100 giây, nhưng thời hạn của script đó áp dụng cho lời gọi engine. Notebook này dùng worker để giới hạn cả dựng mô hình và giải.